## Reading from bronze

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
df.display()

## Silver Transformation

### Trimming the data

In [0]:
query = """
    SELECT
      prd_id,
      prd_key,
      prd_nm,
      COALESCE(prd_cost, 0) AS prd_cost,
      CASE UPPER(TRIM(prd_line))
        WHEN 'R' THEN 'Road'
        WHEN 'S' THEN 'Other Sales'
        WHEN 'M' THEN 'Mountain'
        WHEN 'T' THEN 'Touring'
        ELSE 'n/a'
      END AS prd_line,
      prd_start_dt,
      prd_end_dt
    FROM workspace.bronze.crm_prd_info
"""

df = spark.sql(query)
df.display()

### Normalize the data

In [0]:
query2 = """
    SELECT
      prd_id,
      REPLACE(SUBSTRING(prd_key,1,5), '-','_') AS cat_id,   --EXTRACT Category ID
      SUBSTRING(prd_key, 7, LENGTH(prd_key)) AS prd_key,    --EXTRACT Product Key
      prd_nm,
      COALESCE(prd_cost, 0) AS prd_cost,    --Remove null value from prd_cost
      CASE UPPER(TRIM(prd_line))            --Trim and normalize the prd_line
        WHEN 'R' THEN 'Road'
        WHEN 'S' THEN 'Other Sales'
        WHEN 'M' THEN 'Mountain'
        WHEN 'T' THEN 'Touring'
        ELSE 'n/a'
      END AS prd_line,
      prd_start_dt,
      prd_end_dt
    FROM workspace.bronze.crm_prd_info
"""
df = spark.sql(query2)
df.display()


### Fixing invalid date

In [0]:
query3 = """
    SELECT
      prd_id,
      REPLACE(SUBSTRING(prd_key,1,5), '-','_') AS cat_id,   --EXTRACT Category ID
      SUBSTRING(prd_key, 7, LENGTH(prd_key)) AS prd_key,    --EXTRACT Product Key
      prd_nm,
      COALESCE(prd_cost, 0) AS prd_cost,    --Remove null value from prd_cost
      CASE UPPER(TRIM(prd_line))            --Trim and normalize the prd_line
        WHEN 'R' THEN 'Road'
        WHEN 'S' THEN 'Other Sales'
        WHEN 'M' THEN 'Mountain'
        WHEN 'T' THEN 'Touring'
        ELSE 'n/a'
      END AS prd_line,
      prd_start_dt,
      LEAD(prd_start_dt) OVER(PARTITION BY prd_key ORDER BY prd_start_dt) - INTERVAL '1 Day' AS prd_end_dt
    FROM workspace.bronze.crm_prd_info
"""

df = spark.sql(query3)
df.display()


### Rename table schema

In [0]:
table = "workspace.bronze.crm_prd_info"
rename_schema = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}

for old, new in rename_schema.items():
    df = df.withColumnRenamed(old, new)     # using PySpark

df.display()

### Load transform data into silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_prd_info")

In [0]:
display(spark.table("workspace.silver.crm_prd_info"))